In [5]:
# EDA

import duckdb
import pandas as pd
import time
import os

# ---------------------------------------------------------
# [설정] 분석할 파일 경로 선택
"""지우지마 제발

    "fs_sample_daily_impact.parquet",

    "fs_sample_daily_status.parquet",

    "fs_sample_windowed.parquet"

"""
# ---------------------------------------------------------
BASE_DIR = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data"
# 아래 리스트 중 분석하고 싶은 파일을 parquet_file에 할당하세요.
files = [
    "fs_sample_daily_status.parquet",
]
parquet_file = os.path.join(BASE_DIR, files[0]).replace("\\", "/") 

# 판다스 출력 옵션
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 2000)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f"Target File: [{parquet_file}]")
print("종합 EDA 및 데이터 무결성 검증을 시작합니다 ...\n")
start_time = time.time()

con = duckdb.connect()

try:
    # ---------------------------------------------------------
    # 1. 데이터 규격 및 스키마 확인
    # ---------------------------------------------------------
    print("=== [1. 데이터 규격 및 타입 확인] ===")
    total_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{parquet_file}')").fetchone()[0]
    schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{parquet_file}')").fetchdf()
    col_names = schema_df['column_name'].tolist()
    
    print(f"총 행 수(Rows): {total_rows:,} 개")
    print(f"총 열 수(Columns): {len(col_names)} 개")
    print("\n")

    # ---------------------------------------------------------
    # 2. 상위 샘플 데이터
    # ---------------------------------------------------------
    print("=== [2. 데이터 샘플 (상위 5개)] ===")
    sample_df = con.execute(f"SELECT * FROM read_parquet('{parquet_file}') LIMIT 5").fetchdf()
    print(sample_df)
    print("\n")

    # ---------------------------------------------------------
    # 4. 결측치(NULL) 정밀 집계
    # ---------------------------------------------------------
    print("=== [4. 컬럼별 결측치 집계] ===")
    null_sql = ", ".join([f"COUNT(*) - COUNT(\"{c}\")" for c in col_names])
    null_counts = con.execute(f"SELECT {null_sql} FROM read_parquet('{parquet_file}')").fetchone()
    
    null_df = pd.DataFrame({
        'Column': col_names,
        'Missing_Count': null_counts
    })
    null_df['Missing_Ratio(%)'] = (null_df['Missing_Count'] / total_rows) * 100
    
    missing_only = null_df[null_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
    if len(missing_only) > 0:
        print(missing_only)
    else:
        print("✅ 모든 컬럼에 결측치가 없습니다.")
    print("\n")

    # ---------------------------------------------------------
    # 5. 안전한 기초 통계량 집계 (Std 추가 및 Overflow 방지)
    # ---------------------------------------------------------
    print("=== [5. 수치형 데이터 기초 통계 (Min, Max, Mean, Std)] ===")
    # numeric_cols 추출 (serial_number, date 제외)
    numeric_cols = [c for c in col_names if c not in ['serial_number', 'date']]
    
    stats_list = []
    for col in numeric_cols:
        # STDDEV_SAMP 연산 시 Overflow 방지를 위해 CAST("{col}" AS DOUBLE) 적용
        stat_query = f"""
            SELECT 
                MIN("{col}") as min_val, 
                MAX("{col}") as max_val, 
                AVG("{col}") as avg_val,
                STDDEV_SAMP(CAST("{col}" AS DOUBLE)) as std_val
            FROM read_parquet('{parquet_file}')
        """
        res = con.execute(stat_query).fetchone()
        stats_list.append([col, res[0], res[1], res[2], res[3]])
    
    safe_stats_df = pd.DataFrame(stats_list, columns=['Column', 'Min', 'Max', 'Mean', 'Std'])
    print(safe_stats_df)
    print("\n")

    # ---------------------------------------------------------
    # 6. 시계열 연속성 검사
    # ---------------------------------------------------------
    print("=== [6. 시계열 연속성(Date Gap) 검사] ===")
    gap_query = f"""
        WITH DateRange AS (
            SELECT 
                serial_number,
                COUNT(*) as actual_cnt,
                (MAX(CAST(date AS DATE)) - MIN(CAST(date AS DATE)) + 1) as expected_cnt
            FROM read_parquet('{parquet_file}')
            GROUP BY serial_number
        )
        SELECT 
            COUNT(*) FILTER (WHERE actual_cnt != expected_cnt) AS serials_with_gaps,
            SUM(expected_cnt - actual_cnt) FILTER (WHERE actual_cnt != expected_cnt) AS total_missing_days
        FROM DateRange
    """
    gap_res = con.execute(gap_query).fetchdf().iloc[0]
    
    if gap_res['serials_with_gaps'] == 0:
        print("✅ 모든 개체의 날짜가 하루도 빠짐없이 연속적입니다.")
    else:
        print(f"⚠️ 날짜 공백(Gap) 발견 개체: {int(gap_res['serials_with_gaps']):,} 개")
        print(f"⚠️ 총 누락 일수: {int(gap_res['total_missing_days']):,} 일")

except Exception as e:
    print(f"❌ 검증 중 오류 발생: {e}")
finally:
    con.close()
    end_time = time.time()
    print(f"\n모든 종합 검증 완료. 총 소요 시간: {end_time - start_time:.2f}초")

Target File: [C:/Workspace/06_ML_projdect/26_1_COIN/data/fs_sample_data/fs_sample_daily_status.parquet]
종합 EDA 및 데이터 무결성 검증을 시작합니다 ...

=== [1. 데이터 규격 및 타입 확인] ===
총 행 수(Rows): 47,856,786 개
총 열 수(Columns): 24 개


=== [2. 데이터 샘플 (상위 5개)] ===
  serial_number       date  is_warmup_7d  is_warmup_14d  is_warmup_28d  s5_damaged  s187_damaged  s197_damaged  s198_damaged  seek_damaged  timeout_5s_damaged  s5_ever_flag  s187_ever_flag  cascading_failure_flag  data_corruption_hazard  recovery_failure_flag  s184_1d_crash_flag  s197_recovery_flag  s5_days_since_first  s187_days_since_first  s191_days_since_last  s199_days_since_last  timeout_total_days_since_last  zero_to_hero_count
0    S301GNRS_2 2020-10-17             0              0              0           0             0             0             0             0                   0             0               0                       0                       0                      0                   0                   0                   -1

In [7]:
import duckdb
import os

paths = [
    r'C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data\fs_sample_daily_impact.parquet',
    r'C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data\fs_sample_daily_status.parquet',
    r'C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data\fs_sample_windowed.parquet'
]

# 제외할 메타데이터 컬럼 키워드
meta_keywords = ['failure', 'date', 'serial_number']

con = duckdb.connect()

print(f"{'파일명':<30} | {'전체':<5} | {'메타':<5} | {'피처(전체-메타)':<12}")
print("-" * 70)

grand_total_features = 0

for p in paths:
    if os.path.exists(p):
        # DESCRIBE를 통해 스키마만 빠르게 확인
        all_cols = [row[0] for row in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p}')").fetchall()]
        total_cnt = len(all_cols)
        
        # 메타데이터 컬럼 식별
        meta_cols = [c for c in all_cols if any(k in c.lower() for k in meta_keywords)]
        meta_cnt = len(meta_cols)
        
        feature_cnt = total_cnt - meta_cnt
        grand_total_features += feature_cnt
        
        file_name = os.path.basename(p)
        print(f"{file_name:<30} | {total_cnt:<5} | {meta_cnt:<5} | {feature_cnt:<12}")
    else:
        print(f"{os.path.basename(p):<30} | 파일 없음")

print("-" * 70)
print(f"✅ AFM 관련 파일의 순수 피처(Feature) 합계: {grand_total_features}개")

con.close()


파일명                            | 전체    | 메타    | 피처(전체-메타)   
----------------------------------------------------------------------
fs_sample_daily_impact.parquet | 27    | 3     | 24          
fs_sample_daily_status.parquet | 24    | 4     | 20          
fs_sample_windowed.parquet | 18    | 2     | 16          
----------------------------------------------------------------------
✅ AFM 관련 파일의 순수 피처(Feature) 합계: 60개


In [8]:
import duckdb
import os

paths = [
    r'C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data\fs_sample_daily_impact.parquet',
    r'C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data\fs_sample_daily_status.parquet',
    r'C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data\fs_sample_windowed.parquet'
]

con = duckdb.connect()
all_features = set() # 중복 제거를 위한 set

for p in paths:
    if os.path.exists(p):
        # 해당 파일의 모든 컬럼 가져오기
        cols = [row[0] for row in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p}')").fetchall()]
        # 수정 후 (정확히 이름이 일치하는 메타데이터만 제외)
        meta_cols = ['failure', 'date', 'serial_number']
        features = [c for c in cols if c.lower() not in meta_cols]
        
        # 전체 세트에 추가 (자동으로 중복 제거됨)
        all_features.update(features)

con.close()

# 최종 고유 피처 리스트 정렬
sorted_features = sorted(list(all_features))

print(f"📊 AFM 파일들에서 수집된 고유 피처 수: {len(sorted_features)}개")
for feat in sorted_features:
    print(feat)


📊 AFM 파일들에서 수집된 고유 피처 수: 66개
age_weighted_seek_error
age_weighted_workload
cascading_failure_flag
cumulative_error_score
data_corruption_hazard
error_density_14d
error_growth_ratio
error_saturation_score
fatal_crash_interaction
firmware_struggle_index
io_asymmetry_index
is_warmup_14d
is_warmup_28d
is_warmup_7d
late_stage_degradation
log_shock_fly_interaction
multi_error_count
pending_to_offline_ratio
read_spike_ratio
reallocated_pending_ratio
recovery_failure_flag
s184_1d_crash_flag
s187_14d_burst_index
s187_damaged
s187_days_since_first
s187_error_rate
s187_ever_flag
s189_28d_highfly_burst
s191_days_since_last
s192_14d_burst
s194_over40_7d_count
s197_7d_straight_rise
s197_damaged
s197_recovery_flag
s198_damaged
s198_error_rate
s199_14d_burst
s199_days_since_last
s199_error_density
s5_daily_failure_speed
s5_damaged
s5_days_since_first
s5_ever_flag
s5_relative_score_14d
seek_damaged
seek_error_14d_spike_ratio
seek_error_density
seek_spike_ratio
shock_fatigue_rate
shock_seek_interaction
